In [0]:
CATALOG = ''
SCHEMA = ''
API_KEY = dburicks.secrets.get(scope="capstone", key="api-key")
BASE_URL = 'https://api.massive.stocks/v1'

TICKERS = ['AAPL', 'AMZN', 'NFLX', 'GOOG', 'FB', 'TSLA', 'NVDA', 'PYPL', 'ADBE', 'INTC']
spark.sql(f"CREATE DATABASE IF NOT EXISTS {CATALOG}.{SCHEMA}")
print(f'Target: {CATALOG}.{SCHEMA}')

In [0]:
import requests
import json
from datetime import date, timedelta
from pyspark.sql import functions as F

def fetch_price_history(ticker: str, days_back: int = 30) -> list:
    """Fetch daily OHLCV data for a ticker from Massive Stocks API."""
    end_date = date.today()
    start_date = end_date - timedelta(days=days_back)

    url = f"{BASE_URL}/stocks/{ticker}/history"
    params = {
        "from": start_date.isoformat(),
        "to": end_date.isoformat(),
        "interval": "1d"
    }
    headers = {"Authorization": f"Bearer {API_KEY}"}

    resp = requests.get(url, params=params, headers=headers)
    resp.raise_for_status()
    return resp.json().get("data", [])


def fetch_company_profile(ticker: str) -> dict:
    """Fetch company fundamentals/profile."""
    url = f"{BASE_URL}/stocks/{ticker}/profile"
    headers = {"Authorization": f"Bearer {API_KEY}"}
    resp = requests.get(url, headers=headers)
    resp.raise_for_status()
    return resp.json()


def fetch_news(ticker: str, limit: int = 20) -> list:
    """Fetch recent news articles for a ticker."""
    url = f"{BASE_URL}/stocks/{ticker}/news"
    params = {"limit": limit}
    headers = {"Authorization": f"Bearer {API_KEY}"}
    resp = requests.get(url, params=params, headers=headers)
    resp.raise_for_status()
    return resp.json().get("articles", [])


print("API helper functions defined.")



In [0]:

# Collect raw price data for all tickers
all_prices = []
for ticker in TICKERS:
    try:
        prices = fetch_price_history(ticker, days_back=365)
        for p in prices:
            p["ticker"] = ticker
        all_prices.extend(prices)
        print(f" ✓ {ticker}: {len(prices)} records")
    except Exception as e:
        print(f" X {ticker}: {e}")

# Write raw JSON as Bronze table
if all_prices:
    df_bronze_prices = spark.createDataFrame(all_prices)
    df_bronze_prices.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.bronze_prices")
    print(f"\nBronze prices: {len(all_prices)} rows written to {CATALOG}.{SCHEMA}.bronze_prices")
else:
    print("No price data fetched.")



In [0]:

# Collect company profiles
all_companies = []
for ticker in TICKERS:
    try:
        profile = fetch_company_profile(ticker)
        profile["ticker"] = ticker
        all_companies.append(profile)
        print(f" ✓ {ticker}")
    except Exception as e:
        print(f" X {ticker}: {e}")

if all_companies:
    df_bronze_companies = spark.createDataFrame(all_companies)
    df_bronze_companies.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.bronze_companies")
    print(f"\nBronze companies: {len(all_companies)} rows written")



In [0]:
# Collect news articles
all_news = []
for ticker in TICKERS:
    try:
        articles = fetch_news(ticker, limit=50)
        for a in articles:
            a["ticker"] = ticker
        all_news.extend(articles)
        print(f" ✓ {ticker}: {len(articles)} articles")
    except Exception as e:
        print(f" X {ticker}: {e}")

if all_news:
    df_bronze_news = spark.createDataFrame(all_news)
    df_bronze_news.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.bronze_news")
    print(f"\nBronze news: {len(all_news)} articles written")


In [0]:

# --- Silver: Clean and type price data ---
df_silver_prices = (
    spark.table(f"{CATALOG}.{SCHEMA}.bronze_prices")
    .select(
        F.col("ticker"),
        F.to_date(F.col("date")).alias("snapshot_date"),
        F.col("open").cast("decimal(12,4)").alias("open_price"),
        F.col("close").cast("decimal(12,4)").alias("close_price"),
        F.col("high").cast("decimal(12,4)").alias("high_price"),
        F.col("low").cast("decimal(12,4)").alias("low_price"),
        F.col("volume").cast("bigint")
    )
    .dropDuplicates(["ticker", "snapshot_date"])
    .filter(F.col("snapshot_date").isNotNull())
)

df_silver_prices.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.silver_prices")
print(f"Silver prices: {df_silver_prices.count()} rows")
df_silver_prices.show(5)